# 01 — Data Quality
**FootScout — BHT Berlin**

Mandatory WP: Completeness, duplicates, consistency, outlier analysis.


In [ ]:
import sys, pandas as pd, numpy as np
sys.path.insert(0, "..")
import matplotlib.pyplot as plt, seaborn as sns, plotly.express as px
DATA = "../data/processed/players_merged.csv"
df = pd.read_csv(DATA, low_memory=False)
print(f"Loaded: {len(df):,} players | {df.shape[1]} columns")
df.head(3)

## 1. Completeness — Missing Value Analysis

In [ ]:
missing = (df.isnull().sum()/len(df)*100).sort_values(ascending=False).reset_index()
missing.columns = ["column","missing_pct"]
fig = px.bar(missing[missing.missing_pct>0].head(30), x="column", y="missing_pct",
             title="Missing Rates (%)", color="missing_pct",
             color_continuous_scale="Reds", template="plotly_dark")
fig.update_xaxes(tickangle=-45)
fig.show()
print(f"Columns >20% missing: {(missing.missing_pct>20).sum()}")

## 2. Duplicate Resolution — Mid-Season Transfers

In [ ]:
dups = df[df.duplicated(subset=["player"], keep=False)]
print(f"Duplicate player entries: {len(dups)}")
df_clean = (df.sort_values("playing_time_min", ascending=False)
              .drop_duplicates(subset=["player"], keep="first")
              .reset_index(drop=True))
print(f"After dedup: {len(df_clean):,} unique players")

## 3. Consistency — Bounds Validation

In [ ]:
BOUNDS = {"playing_time_min":(450,4000), "age":(15,45),
          "pass_completion_pct":(0,100), "gls_per90":(0,5)}
for col,(lo,hi) in BOUNDS.items():
    if col in df_clean.columns:
        s = pd.to_numeric(df_clean[col], errors="coerce")
        n = ((s<lo)|(s>hi)).sum()
        print(f"  {col}: {n} violations (bounds: [{lo}, {hi}])")

## 4. Outlier Analysis — IQR Visualization

In [ ]:
STAT_COLS = [c for c in ["gls_per90","ast_per90","xg_per90","tackles_tkl_per90"] if c in df_clean.columns]
fig, axes = plt.subplots(1, len(STAT_COLS), figsize=(4*len(STAT_COLS), 4))
fig.patch.set_facecolor("#0F1117")
for ax, col in zip(axes if len(STAT_COLS)>1 else [axes], STAT_COLS):
    data = pd.to_numeric(df_clean[col], errors="coerce").dropna()
    Q1,Q3 = data.quantile([0.25,0.75])
    IQR = Q3-Q1; lo,hi = Q1-1.5*IQR, Q3+1.5*IQR
    ax.boxplot(data, patch_artist=True, boxprops=dict(facecolor="#6C63FF",alpha=0.5))
    ax.axhline(lo,color="#FF6B6B",linestyle="--",linewidth=1.5)
    ax.axhline(hi,color="#FF6B6B",linestyle="--",linewidth=1.5,label="IQR fence")
    ax.set_title(col.replace("_per90","/90"),color="white"); ax.set_facecolor("#111827")
    ax.tick_params(colors="white")
    ax.set_xlabel(f"{((data<lo)|(data>hi)).sum()} outliers",color="#FF6B6B",fontsize=8)
plt.suptitle("IQR Outlier Analysis",color="white"); plt.tight_layout(); plt.show()